# Lineare Regression

**Ziel:** Lineare Regression Schritt für Schritt verstehen:

1. **Theorie** – Modell und Methode der kleinsten Quadrate
2. **Händisch** – mit Schiebereglern Steigung & Achsenabschnitt selbst suchen
3. **Automatisch** – mit `scikit-learn` die optimalen Parameter finden
4. **Übung** – auf echte Daten anwenden (mit Train/Test-Split)


## 1. Mathematische Grundlagen

### Modell

Eine Gerade mit Steigung $\beta_1$ und Achsenabschnitt $\beta_0$:

$$\hat{y} = \beta_0 + \beta_1 \cdot x$$

- $\hat{y}$: Vorhersage
- $x$: Eingangsvariable
- $\beta_0$: y-Achsenabschnitt (Intercept)
- $\beta_1$: Steigung (Slope)

### Methode der kleinsten Quadrate

Wir suchen $\beta_0, \beta_1$ so, dass die Summe der quadrierten Fehler minimal wird:

$$\min_{\beta_0,\, \beta_1} \sum_{i=1}^{n} \bigl(y_i - \hat{y}_i\bigr)^2
= \sum_{i=1}^{n} \bigl(y_i - \beta_0 - \beta_1 x_i\bigr)^2$$

Die analytische Lösung lautet:

$$\beta_1 = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n}(x_i - \bar{x})^2}
= \frac{\mathrm{Cov}(x, y)}{\mathrm{Var}(x)}$$

$$\beta_0 = \bar{y} - \beta_1 \cdot \bar{x}$$

### Gütemaße

**Mittlerer quadratischer Fehler (MSE):**

$$\mathrm{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Bestimmtheitsmaß $R^2$:**

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

- $R^2 = 1$: perfekte Vorhersage
- $R^2 = 0$: Modell ist nicht besser als der Mittelwert


In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

np.random.seed(42)


## 2. Händisch fitten mit Schiebereglern

Wir erzeugen eine Punktwolke und versuchen, mit den Schiebereglern für
**Steigung $m$** und **Achsenabschnitt $b$** eine möglichst gute Gerade zu finden.

Die **gestrichelten Linien** zeigen die Residuen (Abstände der Punkte zur Geraden).
Oben steht der aktuelle **MSE** – je kleiner, desto besser.

Beide Slider lassen sich frei kombinieren.


In [3]:
# Punktwolke erzeugen
np.random.seed(42)
n_demo = 25
x_demo = np.linspace(0, 10, n_demo)
m_wahr, b_wahr = 2.5, 5.0
y_demo = m_wahr * x_demo + b_wahr + np.random.normal(0, 3, n_demo)

print(f"Anzahl Punkte: {n_demo}")
print(f"Wahre Parameter: m = {m_wahr}, b = {b_wahr}")


Anzahl Punkte: 25
Wahre Parameter: m = 2.5, b = 5.0


In [ ]:
import ipywidgets as widgets
from IPython.display import display

x_range = [float(x_demo.min() - 0.5), float(x_demo.max() + 0.5)]
y_range = [float(y_demo.min() - 5), float(y_demo.max() + 5)]

out = widgets.Output()

def plot_gerade(m, b):
    y_line = m * x_demo + b
    err = float(np.mean((y_demo - y_line) ** 2))

    fig = go.Figure()
    fig.add_scatter(x=x_demo, y=y_demo, mode='markers', name='Datenpunkte',
                    marker=dict(color='#2196F3', size=8))
    fig.add_scatter(x=x_demo, y=y_line, mode='lines', name='Gerade',
                    line=dict(color='#EF5350', width=2))
    for xi, yi, ypi in zip(x_demo, y_demo, y_line):
        fig.add_shape(type='line',
                      x0=float(xi), x1=float(xi),
                      y0=float(yi), y1=float(ypi),
                      line=dict(color='gray', width=1, dash='dot'))
    fig.update_xaxes(title_text='x', range=x_range)
    fig.update_yaxes(title_text='y', range=y_range)
    fig.update_layout(
        title=dict(text=f'm = {m:.2f},  b = {b:.2f}  →  MSE = {err:.2f}',
                   x=0.5, xanchor='center'),
        height=500, margin=dict(t=60, b=40, l=60, r=20),
    )

    out.clear_output(wait=True)
    with out:
        display(fig)

slider_m = widgets.FloatSlider(value=1.0, min=-2.0, max=6.0, step=0.1,
                               description='Steigung m:',
                               continuous_update=False,
                               layout=widgets.Layout(width='95%'),
                               style={'description_width': '140px'})
slider_b = widgets.FloatSlider(value=0.0, min=-10.0, max=20.0, step=0.5,
                               description='Achsenabschnitt b:',
                               continuous_update=False,
                               layout=widgets.Layout(width='95%'),
                               style={'description_width': '140px'})

widgets.interactive_output(plot_gerade, {'m': slider_m, 'b': slider_b})

display(slider_m, slider_b, out)
plot_gerade(slider_m.value, slider_b.value)

FloatSlider(value=1.0, continuous_update=False, description='Steigung m:', layout=Layout(width='95%'), max=6.0…

FloatSlider(value=0.0, continuous_update=False, description='Achsenabschnitt b:', layout=Layout(width='95%'), …

Output()

## 3. Automatisch fitten mit scikit-learn

`scikit-learn` löst die Methode der kleinsten Quadrate analytisch –
also genau das, was wir oben händisch gesucht haben.


In [4]:
# sklearn braucht X als 2D-Array (n_samples, n_features)
X_demo = x_demo.reshape(-1, 1)

modell = LinearRegression()
modell.fit(X_demo, y_demo)

m_fit = modell.coef_[0]
b_fit = modell.intercept_
y_pred = modell.predict(X_demo)

print("=== Gefundene Parameter ===")
print(f"Steigung m:        {m_fit:.4f}   (wahrer Wert: {m_wahr})")
print(f"Achsenabschnitt b: {b_fit:.4f}   (wahrer Wert: {b_wahr})")
print()
print("=== Modellgüte ===")
print(f"MSE: {mean_squared_error(y_demo, y_pred):.4f}")
print(f"R²:  {r2_score(y_demo, y_pred):.4f}")


=== Gefundene Parameter ===
Steigung m:        2.1152   (wahrer Wert: 2.5)
Achsenabschnitt b: 6.4336   (wahrer Wert: 5.0)

=== Modellgüte ===
MSE: 6.5686
R²:  0.8601


In [5]:
# Plot: Datenpunkte + automatisch gefundene Regressionsgerade
fig = go.Figure()
fig.add_scatter(x=x_demo, y=y_demo, mode='markers',
                marker=dict(color='#2196F3', size=8),
                name='Datenpunkte')
fig.add_scatter(x=x_demo, y=y_pred, mode='lines',
                line=dict(color='#EF5350', width=2),
                name=f'sklearn:  y = {m_fit:.2f}·x + {b_fit:.2f}')

# Residuen als senkrechte Linien einzeichnen
for xi, yi, ypi in zip(x_demo, y_demo, y_pred):
    fig.add_shape(type='line', x0=xi, x1=xi, y0=yi, y1=ypi,
                  line=dict(color='gray', width=1, dash='dot'))

fig.update_layout(title='Regressionsgerade mit Residuen',
                  xaxis_title='x', yaxis_title='y',
                  height=500)
fig.show()


## 4. Übung: Regression auf echten Daten

Wir wenden lineare Regression auf echte Messwerte an:
PV-Leistung einer kleinen Anlage gegen Globalstrahlung — beides 15-min UTC,
Jahr 2023 in Stuttgart.

| Datei | Inhalt | Quelle |
|---|---|---|
| `pv_data_year_stuttgart_0.6kwp.csv` | PV-Leistung [W] — Spalte `pv_w` | **PV-Live** (Fraunhofer ISE), Station Stuttgart, Si-Referenzzelle Süd 25° tilt, skaliert auf 600 Wp BKW × η=0.85 — DOI [10.5281/zenodo.17079486](https://doi.org/10.5281/zenodo.17079486), CC-BY-4.0 |
| `historical_weather_forecast_radiation_stuttgart.csv` | Globalstrahlung [W/m²] — Spalte `ghi_wm2` | **Open-Meteo Historical-Forecast** (archivierte DWD-ICON / AROME-Vorhersagen), gleiche Lat/Lon — [open-meteo.com](https://open-meteo.com), CC-BY-4.0 |

Wir bereiten dir **Daten-Laden + Filter + Train/Test-Split** unten schon vor.
Deine Aufgabe sind die drei Regressions-Schritte am Ende.


In [6]:
# Daten laden und joinen (vorbereitet)
from pathlib import Path

pv  = pd.read_csv("pv_data_year_stuttgart_0.6kwp.csv",
                  comment="#", parse_dates=["datetime_utc"], index_col="datetime_utc")
rad = pd.read_csv("historical_weather_forecast_radiation_stuttgart.csv",
                  comment="#", parse_dates=["datetime_utc"], index_col="datetime_utc")

df = pd.concat([rad["ghi_wm2"], pv["pv_w"]], axis=1).dropna()
print(f"{len(df)} Zeilen geladen ({df.index[0].date()} → {df.index[-1].date()})")
df.head()


35040 Zeilen geladen (2023-01-01 → 2023-12-31)


,ghi_wm2,pv_w
datetime_utc,,
2023-01-01 00:00:00+00:00,0.0,0.0
2023-01-01 00:15:00+00:00,0.0,0.0
2023-01-01 00:30:00+00:00,0.0,0.0
2023-01-01 00:45:00+00:00,0.0,0.0
2023-01-01 01:00:00+00:00,0.0,0.0


In [7]:
# Beide Zeitreihen für eine Beispielwoche im Juni anschauen (zwei y-Achsen)
woche = df.loc["2023-06-19":"2023-06-25"]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(x=woche.index, y=woche["pv_w"], name="PV-Leistung [W]",
                line=dict(color="#EF5350", width=1.5), secondary_y=False)
fig.add_scatter(x=woche.index, y=woche["ghi_wm2"], name="GHI [W/m²]",
                line=dict(color="#2196F3", width=1.5, dash="dot"), secondary_y=True)

fig.update_yaxes(title_text="PV-Leistung [W]", secondary_y=False, color="#EF5350")
fig.update_yaxes(title_text="GHI [W/m²]",      secondary_y=True,  color="#2196F3")
fig.update_layout(title="Beispielwoche 19.–25. Juni 2023 (Stuttgart, UTC)",
                  height=400, margin=dict(t=60, b=40, l=60, r=60))
fig.show()


In [8]:
# Filter Tagstunden + 80/20 Train/Test-Split (vorbereitet)
df_day = df[df["ghi_wm2"] > 5]

X = df_day[["ghi_wm2"]].values
y = df_day["pv_w"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training: {len(X_train)} Punkte")
print(f"Test:     {len(X_test)} Punkte")


Training: 13732 Punkte
Test:     3433 Punkte


### Deine Aufgabe

Drei Schritte — analog zu Sektion 3, nur mit echten Daten:


In [9]:
# Schritt 1: LinearRegression trainieren
modell_pv = LinearRegression()
modell_pv.fit(X_train, y_train)

m_pv = modell_pv.coef_[0]
b_pv = modell_pv.intercept_

print(f"Steigung m:        {m_pv:.4f}")
print(f"Achsenabschnitt b: {b_pv:.2f}")


Steigung m:        0.5729
Achsenabschnitt b: -0.64


In [10]:
# Schritt 2: R² und MSE getrennt für Training und Test
y_pred_train = modell_pv.predict(X_train)
y_pred_test  = modell_pv.predict(X_test)

print(f"{'Metrik':<8s} {'Training':>12s} {'Test':>12s}")
print("-" * 36)
print(f"{'R²':<8s} {r2_score(y_train, y_pred_train):>12.4f} "
      f"{r2_score(y_test, y_pred_test):>12.4f}")
print(f"{'MSE':<8s} {mean_squared_error(y_train, y_pred_train):>12.2f} "
      f"{mean_squared_error(y_test, y_pred_test):>12.2f}")


Metrik       Training         Test
------------------------------------
R²             0.7937       0.7885
MSE           4589.41      4676.35


In [11]:
# Schritt 3: Plot — Sample von 2000 Punkten je Set, sonst zu träge
np.random.seed(0)
idx_tr = np.random.choice(len(X_train), size=min(2000, len(X_train)), replace=False)
idx_te = np.random.choice(len(X_test),  size=min(500,  len(X_test)),  replace=False)

x_line = np.linspace(0, X.max(), 100).reshape(-1, 1)
y_line = modell_pv.predict(x_line)

fig = go.Figure()
fig.add_scatter(x=X_train[idx_tr, 0], y=y_train[idx_tr], mode="markers",
                marker=dict(color="#2196F3", size=4, opacity=0.4),
                name=f"Training (Sample {len(idx_tr)})")
fig.add_scatter(x=X_test[idx_te, 0], y=y_test[idx_te], mode="markers",
                marker=dict(color="#FF9800", size=5, opacity=0.6, symbol="diamond"),
                name=f"Test (Sample {len(idx_te)})")
fig.add_scatter(x=x_line.flatten(), y=y_line, mode="lines",
                line=dict(color="#EF5350", width=2),
                name=f"Regression:  y = {m_pv:.3f}·x + {b_pv:.1f}")
fig.update_layout(title="Lineare Regression auf echten PV-Daten",
                  xaxis_title="GHI [W/m²]", yaxis_title="PV-Leistung [W]",
                  height=500, margin=dict(t=50, b=40, l=60, r=20))
fig.show()
